# Chapter 06 — Neural Networks & Deep Learning

Tensor와 Autograd를 활용해 Neural Network를 정의하고 학습하는 실습입니다. 개념 설명과 프로젝트 요구사항은 Chapter 06의 `README.md`에 정리되어 있습니다. 모든 셀은 CPU에서 실행할 수 있습니다.

## 1. 실행 환경과 Tensor

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
torch.set_num_threads(1)
print('PyTorch:', torch.__version__)
print('Device: cpu')

## 2. Neuron과 Linear Layer
가중합 $z=wx+b$와 Linear Layer의 Weight Shape를 비교합니다.

In [ ]:
x = torch.tensor([2., 3.])
w = torch.tensor([0.5, -1.])
z = torch.dot(x, w) + 1.0
print('z:', z.item(), 'ReLU(z):', torch.relu(z).item())
layer = nn.Linear(3, 2)
print('Weight:', tuple(layer.weight.shape), 'Batch output:', tuple(layer(torch.zeros(4, 3)).shape))

## 3. Activation Functions
ReLU, Sigmoid, Tanh의 출력 범위를 확인합니다.

In [ ]:
values = torch.linspace(-4, 4, 200)
plt.figure(figsize=(7, 4))
plt.plot(values, torch.relu(values), label='ReLU')
plt.plot(values, torch.sigmoid(values), label='Sigmoid')
plt.plot(values, torch.tanh(values), label='Tanh')
plt.legend(); plt.grid(alpha=0.2); plt.title('Activation Functions'); plt.show()

## 4. nn.Module과 Forward
모델은 `__init__()`에 Layer를 등록하고 `forward()`에서 계산 순서를 정의합니다.

In [ ]:
class MyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 3))

    def forward(self, x):
        return self.layers(x)

network = MyNetwork()
print(network)
print('Logits Shape:', tuple(network(torch.randn(5, 2)).shape))
print('Parameter count:', sum(p.numel() for p in network.parameters()))

## 5. Softmax와 CrossEntropyLoss
CrossEntropyLoss에는 Softmax 결과가 아니라 원본 Logits를 전달합니다.

In [ ]:
logits = torch.tensor([[2., 0.5, -1.], [0.1, -0.2, 1.2]])
labels = torch.tensor([0, 2], dtype=torch.int64)
probs = logits.softmax(dim=1)
loss = nn.CrossEntropyLoss()(logits, labels)
print('Probs:', probs.round(decimals=3), 'Loss:', round(loss.item(), 4))

## 6. Backpropagation
계산 그래프에 따라 Gradient가 자동으로 구해지는지 확인합니다.

In [ ]:
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
prediction = w * 2 + b
loss = (prediction - 5)**2
loss.backward()
print('Loss:', loss.item(), 'dL/dw:', w.grad.item(), 'dL/db:', b.grad.item())

## 7. 첫 번째 Training Loop
간단한 회귀 모델로 `zero_grad → forward → loss → backward → step`을 연결합니다.

In [ ]:
torch.manual_seed(9)
x_train = torch.linspace(-1, 1, 80).unsqueeze(1)
y_train = 0.7 * x_train + 0.2
loader = DataLoader(TensorDataset(x_train, y_train), batch_size=16, shuffle=True)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.15)
history=[]
for epoch in range(30):
    model.train()
    loss_total=0.
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        train_loss=nn.MSELoss()(model(batch_x), batch_y)
        train_loss.backward(); optimizer.step()
        loss_total += train_loss.item() * len(batch_x)
    history.append(loss_total / len(x_train))
print('MSE first:', round(history[0], 5), 'last:', round(history[-1], 5))
plt.figure(figsize=(6,3)); plt.plot(history); plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.show()

## 8. Train/Eval Mode와 추론

In [ ]:
model.eval()
with torch.inference_mode():
    print('Pred at x=0:', round(model(torch.zeros(1, 1)).item(), 4))

dropout_model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Dropout(0.5), nn.Linear(8, 1))
dropout_model.eval()
print('Dropout training mode:', dropout_model[2].training)

## 9. 비선형 Classification 실습 — XOR
외부 CSV 다운로드 없이 같은 규칙으로 데이터를 만듭니다.

In [ ]:
rng=np.random.default_rng(42)
samples=[]; targets=[]
for a in (-1, 1):
    for b in (-1, 1):
        for _ in range(40):
            samples.append([a*(1+rng.normal(0,.2)), b*(1+rng.normal(0,.2))])
            targets.append(float(a!=b))
inputs = torch.tensor(np.asarray(samples), dtype=torch.float32)
targets = torch.tensor(targets, dtype=torch.float32)
print('XOR Shape:',tuple(inputs.shape), tuple(targets.shape))

In [ ]:
torch.manual_seed(7)
xor_model=nn.Sequential(nn.Linear(2,8), nn.Tanh(), nn.Linear(8,1))
optimizer=torch.optim.Adam(xor_model.parameters(), lr=.025)
for epoch in range(230):
    optimizer.zero_grad()
    loss = nn.BCEWithLogitsLoss()(xor_model(inputs).squeeze(1), targets)
    loss.backward(); optimizer.step()
xor_model.eval()
with torch.inference_mode():
    predictions=(xor_model(inputs).squeeze(1).sigmoid() >= .5).float()
print('XOR training Accuracy:', (predictions==targets).float().mean().item())

위 XOR 실습은 Training Data에서의 정확도를 확인하는 예제입니다. 실제 성능을 평가할 때는 Train/Test를 분리해야 합니다. 분할과 모델 비교는 Mini Project 01에서 수행합니다.

## 10. 저장과 복원
Notebook에서는 임시 폴더에 Weight를 저장하고 다시 불러옵니다.

In [ ]:
from pathlib import Path
import tempfile
with tempfile.TemporaryDirectory() as temp:
    weight_path=Path(temp)/'weights.pt'
    torch.save(xor_model.state_dict(), weight_path)
    restored=nn.Sequential(nn.Linear(2,8), nn.Tanh(), nn.Linear(8,1))
    restored.load_state_dict(torch.load(weight_path, weights_only=True, map_location='cpu'))
    restored.eval()
    with torch.inference_mode():
        assert torch.allclose(xor_model(inputs), restored(inputs))
    print('Restored model predictions match.')

## 11. 학습 결과 점검
Batch, Output, Target의 Shape를 비교하고 Loss가 유한한지 확인합니다.

In [ ]:
with torch.inference_mode():
    output=xor_model(inputs)
print('Input:',tuple(inputs.shape),'Logit:',tuple(output.shape),'Label:',tuple(targets.shape))
print('All logits finite:', bool(torch.isfinite(output).all()))

## 학습 정리

- Neural Network는 Linear Layer 사이에 비선형 Activation Function을 넣어 다양한 관계를 표현합니다.
- `forward()`는 입력의 계산 흐름을 정의합니다.
- Loss 함수의 입력은 문제 유형에 따라 다르며, CrossEntropy/BCEWithLogits에는 Logit을 전달합니다.
- `backward()`는 Gradient를 계산하고, Optimizer는 Parameter를 갱신합니다.
- 실제 모델의 성능은 분리된 Validation/Test에서 확인해야 합니다.

프로젝트별 상세 요구사항과 모범답안은 Chapter 06의 `README.md`를 참고하세요.